In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
wine = load_wine()

df = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

df["target"] = wine.target

print("Первые 5 строк датасета:")
display(df.head())

print("\nРазмер датасета:", df.shape)
print("\nКоличество классов:")
print(df["target"].value_counts())

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print("Количество признаков:", X.shape[1])
print("Количество объектов:", X.shape[0])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Масштабирование выполнено.")

In [ ]:
pca = PCA(n_components=2)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Размер после PCA:", X_train_pca.shape)

print(
    "Объяснённая дисперсия:",
    pca.explained_variance_ratio_
)

print(
    "Общая объяснённая дисперсия:",
    round(sum(pca.explained_variance_ratio_) * 100, 2),
    "%"
)

In [ ]:
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    X_train_pca[:, 0],
    X_train_pca[:, 1],
    c=y_train,
    cmap="viridis",
    edgecolors="black",
    alpha=0.8
)

plt.xlabel("First Principal Component")
plt.ylabel("Second Principal Component")
plt.title("PCA Components Visualization")

plt.colorbar(
    scatter,
    label="Wine Class"
)

plt.grid(True)
plt.show()

In [ ]:
components_list = [2, 5, 10]

accuracy_results = []

for n in components_list:

    pca = PCA(n_components=n)

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_pca, y_train)

    predictions = model.predict(X_test_pca)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    accuracy_results.append(accuracy)

    print(
        f"Количество компонент: {n} | "
        f"Accuracy: {accuracy:.4f}"
    )

In [ ]:
results_df = pd.DataFrame({
    "PCA Components": components_list,
    "Accuracy": accuracy_results
})

display(results_df)

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    components_list,
    accuracy_results,
    marker="o",
    linewidth=2
)

plt.xlabel("Number of PCA Components")
plt.ylabel("Classification Accuracy")
plt.title("PCA Accuracy vs Components")

plt.xticks(components_list)
plt.grid(True)

plt.savefig(
    "pca_accuracy_results.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("График сохранён: pca_accuracy_results.png")